# 🏙️ The Chinese Decentralization Problem Set  
*(based on Baum-Snow et al., 2017)*

## Overview

This problem set explores how urban infrastructure—especially **radial roads and ring roads**—has shaped **urban population growth in China** between 1990 and 2010.  
Our data are derived from *Baum-Snow, Brandt, Henderson, Turner, and Zhang (2017), “Does Urban Rail Transit Investment Encourage Urban Growth? Evidence from China,”* NBER Working Paper 24596.

The dataset, `china_decent_regs.parquet`, contains variables at the **prefecture-city** level that measure changes in population, GDP, and infrastructure across time.  
We will replicate **Table 4** from the paper using Python, which reports **instrumental variables (IV) regressions** of population growth on road infrastructure.

---

## Learning Goals

By completing this exercise, you will:

1. Reinforce how to specify and interpret an **instrumental variables regression** in Python.  
2. Learn how to identify **endogenous** regressors and their corresponding **instruments**.  
3. Gain additional experience with **clustered standard errors** and **first-stage diagnostics** (e.g., weak instrument tests).  

---

## Core Research Question

> **Did the expansion of radial and ring roads cause faster population growth in Chinese cities?**

Formally, we estimate models of the form:

$$
\Delta \ln(\text{Population}_{i,1990\!-\!2010}) = 
\alpha + \beta \cdot \text{Roads}_{i,2010} + X_i'\gamma + \varepsilon_i,
$$

where:
- $i$ indexes prefecture-level cities,
- $\text{Roads}_{i,2010}$ measures the extent of roads by 2010,
- $X_i$ is a vector of baseline controls,
- $\varepsilon_i$ is an error term clustered by province.

Because road placement may be **endogenous** (e.g., more developed cities built more roads), we instrument for modern road measures using **historical road plans from 1962**:

$$
\text{Roads}_{i,2010} = \pi_0 + \pi_1 \cdot \text{Roads}_{i,1962} + X_i'\rho + u_i.
$$

The coefficient $\beta$ from the second stage represents the **causal effect** of road infrastructure on urban population growth.

---

## The Data

The dataset `china_decent_regs.parquet` (≈280 observations) is the product of extensive preparation combining:
- Historical maps of roads and railways (1924–2010),
- Census data (1982–2010),
- GDP and employment by sector,
- Geographic and institutional indicators.

---

### Relevant Variables

- `D_censuspop9010_cp` — Log change in core-prefecture (CP) population, 1990–2010  
- `D_censuspop9010_pref` — Log change in prefecture population, 1990–2010  
- `all_road_2010_rays` — Number of radial road connections by 2010  
- `road_1962_rays` — Number of planned radial road connections (1962)  
- `rail_2010_rays` — Number of radial railway connections by 2010  
- `rail_1962_rays` — Number of planned railway connections (1962)  
- `po_s_all_road_2010_ringxda` — Dummy = 1 if ring roads existed by 2010  
- `po_s_road_1962_ringxda` — Dummy = 1 if ring roads were planned in 1962  
- `lall_road_2010_km_pfcp` — Log kilometers of roads in the prefecture minus core area, 2010  
- `lroad_1962_km_pfcp` — Log kilometers of roads in the prefecture minus core area, 1962  
- `larea_cc90` — Log area of the central city (1990)  
- `larea_pf05` — Log area of the prefecture (2005)  
- `province_capitalPlus` — Dummy = 1 if provincial capital or provincial-level city  
- `lcensuspop1982_pref` — Log prefecture population, 1982  
- `frac_highedu1982_pref` — Fraction of adults with high school or higher education, 1982  
- `share_emp_man1982_pref` — Share of manufacturing employment, 1982  
- `ruralMigp00` — Share of rural migrants around 2000  
- `province05` — Province identifier (used for clustering)  
- `D0` — Sample inclusion dummy (1 = included in analysis)

---

**Next:** In the following code cells, you’ll load `china_decent_regs.parquet`, inspect key variables, and begin estimating the IV regressions.


# Question 1: Setup and Data Inspection
 
In this first part of the assignment, you will:
1. Import the necessary Python packages,
2. Define your base working directory,
3. Load the dataset `china_decent_regs.parquet`, and
4. Inspect the structure and summary statistics of the data.


In [1]:
# QUESTION 1A — Setup and Data Inspection

# load libraries os, pandas, numpy, statsmodels.api as sm, IV2SLS from linearmodels.iv
import os
from pathlib import Path
import pandas as pd
import numpy as np
import statsmodels.api as sm
try:
    from linearmodels.iv import IV2SLS
except ImportError:
    import pip
    print('installing \'linearmodels\'')
    pip.main(['install', 'linearmodels'])
    from linearmodels.iv import IV2SLS

# Display settings
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)

# Define base directory (adjust this path as needed)
print(os.getcwd())  # Print current working directory to help set the correct path
base_dir = Path("H:/Documents/GradSchool/UrbanEcon/Assignments")

# Load dataset
china_df = pd.read_parquet(base_dir / "china_decent_regs.parquet")

# Inspect structure
print(china_df.info())
print(china_df.head())

# Summary statistics for key variables
summary_vars = [
    "D_censuspop9010_cp", "all_road_2010_rays", "road_1962_rays",
    "larea_cc90", "larea_pf05", "province_capitalPlus",
    "lcensuspop1982_pref", "frac_highedu1982_pref",
    "share_emp_man1982_pref", "ruralMigp00"
]
summary_stats = china_df[summary_vars].describe()
print(summary_stats)

h:\Documents\GradSchool\UrbanEcon\Assignments
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 286 entries, 0 to 285
Columns: 2888 entries, cc05 to D0
dtypes: float32(565), float64(1471), int16(20), int32(580), int8(250), object(2)
memory usage: 4.5+ MB
None
     cc05  gdp_prefb1990  gdp_sector1_prefb1990  gdp_sector2_prefb1990  gdp_sector3_prefb1990  tot_pop_prefb1990  \
0  110000            NaN                    NaN                    NaN                    NaN                NaN   
1  120000            NaN                    NaN                    NaN                    NaN             870.46   
2  130100            NaN                    NaN                    NaN                    NaN                NaN   
3  130200            NaN                    NaN                    NaN                    NaN                NaN   
4  130300            NaN                    NaN                    NaN                    NaN                NaN   

   a101_prefb1990  a105_prefb1990  a106_pre

# Question 1B — Filter Relevant Variables

The raw dataset contains nearly 3,000 columns created for multiple tables in the original paper.  
For this problem set, we only need a small subset of variables used in Table 4.

In this step, you will:
1. Define the list of relevant variables (outcome, instruments, controls, and identifiers),  
2. Filter the DataFrame to keep only those variables, and  
3. Drop any observations missing the main outcome variable `D_censuspop9010_cp`.
4. Finally, keep ONLY the analysis sample where D0=1

After filtering, the dataset will be much smaller and easier to inspect.

In [6]:
# QUESTION 1B — Filter dataset to relevant variables

# Define the relevant variables for the Chinese Decentralization Problem Set (Table 4)
vars_needed = [
    # Outcome and instruments
    "D_censuspop9010_cp", "D_censuspop9010_pref",
    "all_road_2010_rays", "road_1962_rays",
    "rail_2010_rays", "rail_1962_rays",
    "po_s_all_road_2010_ringxda", "po_s_road_1962_ringxda",
    "lall_road_2010_km_pfcp", "lroad_1962_km_pfcp",
    
    # Controls
    "larea_cc90", "larea_pf05", "province_capitalPlus",
    "lcensuspop1982_pref", "frac_highedu1982_pref",
    "share_emp_man1982_pref", "ruralMigp00",
    
    # Clustering and sample
    "province05", "D0"
]

# Keep only those columns
working_china_df = china_df[vars_needed].copy()

# Drop rows missing the main outcome variable
working_china_df = working_china_df.dropna(subset=["D_censuspop9010_cp"])

# Keep analysis sample only
working_china_df = working_china_df[working_china_df["D0"] == 1]
print(f"{working_china_df.shape=}")
working_china_df.info() # Check the structure of the filtered dataset

working_china_df.shape=(257, 19)
<class 'pandas.core.frame.DataFrame'>
Index: 257 entries, 0 to 283
Data columns (total 19 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   D_censuspop9010_cp          257 non-null    float32
 1   D_censuspop9010_pref        257 non-null    float32
 2   all_road_2010_rays          257 non-null    int8   
 3   road_1962_rays              257 non-null    int8   
 4   rail_2010_rays              257 non-null    int8   
 5   rail_1962_rays              257 non-null    int8   
 6   po_s_all_road_2010_ringxda  257 non-null    float32
 7   po_s_road_1962_ringxda      257 non-null    float32
 8   lall_road_2010_km_pfcp      257 non-null    float32
 9   lroad_1962_km_pfcp          257 non-null    float32
 10  larea_cc90                  257 non-null    float32
 11  larea_pf05                  257 non-null    float32
 12  province_capitalPlus        257 non-null    float32
 13  lcensus

# Question 2 — OLS Baseline Regressions (Replicate Table 3)

In this section, you will estimate three OLS models that correspond to columns (1), (2), and (6) of Table 3 in Baum-Snow et al. (2017).

Each regression uses **Δln(CC Pop) 1990–2010** as the dependent variable:

1. **Column (1)** — Regress Δln(CC Pop) on **2010 radial highways** (no controls).  
2. **Column (2)** — Add **base controls** and **Δln(Pref Pop)** to account for prefecture-level population growth.  
3. **Column (6)** — Keep all controls and include the **2010 ring-road indicator** to test for additional effects of orbital infrastructure.

Use the already-filtered dataset (`df`, which includes only `D0 == 1`) and **cluster standard errors by province**.

After running the regressions, compare your coefficients with those in Table 3.  
- How does including Δln(Pref Pop) affect the coefficient on radial highways?  
- Does adding the ring-road indicator change the results substantially?


In [3]:
# QUESTION 2 — OLS Baseline Regressions (Replicate Table 3)

import statsmodels.formula.api as smf # if you wish to use formula API

# (1) Roads only
# your code here

# (2) Add base controls + Δln(Pref Pop)
# your code here

# (6) Add ring-road indicator
# your code here

# Display compact coefficient tables
# your code here


# Question 3 — Instrumental Variables (2SLS)

In the previous question, we estimated the relationship between road infrastructure and population growth using **ordinary least squares (OLS)**.  
However, OLS estimates may be **biased** if roads were built *because* cities were already growing.  
To address this potential **endogeneity**, we now estimate **instrumental-variables (IV)** regressions using **two-stage least squares (2SLS)**.

---

## 3.1 The IV setup

In a 2SLS model we distinguish among:
- **Dependent variable** – `y`
- **Exogenous regressors** – `x`
- **Endogenous regressors** – `z`
- **Instruments** – `w`

Formally:

$$
y = \alpha + \beta z + x'\gamma + \varepsilon, \qquad
z = \pi_0 + \pi_1 w + x'\rho + u
$$

The key requirement is that instruments `w` are:
1. **Relevant** — correlated with the endogenous variable `z`, and  
2. **Exogenous** — uncorrelated with the structural error \(\varepsilon\).

---

## 3.2 Using `IV2SLS` in Python

We use the `linearmodels` package, which implements 2SLS estimation as:

```python
from linearmodels.iv import IV2SLS
import statsmodels.api as sm

# Example setup
y = df["y"]                       # dependent variable
exog = sm.add_constant(df[["x1", "x2"]])   # exogenous regressors (with constant)
endog = df[["z1"]]                # endogenous variable(s)
instr = df[["w1"]]                # instrument(s)

# Fit the IV model with clustered standard errors
iv_model = IV2SLS(y, exog, endog, instr).fit(
    cov_type="clustered", clusters=df["cluster_id"]
)

# Display both second stage and first stage results
print(iv_model.summary)
print(iv_model.first_stage.summary)
```

---

## 3.3 Tasks

1. **Estimate three IV specifications** that correspond to the OLS regressions from Question 2.  
   - Treat modern road measures as **endogenous** and instrument them using **historical road variables**.  
   - Remember to **cluster standard errors by province**.

2. **Print both outputs** for each model:  
   - The **second-stage regression results**  
     ```python
     print(iv_model.summary)
     ```  
   - The **first-stage regression results**  
     ```python
     print(iv_model.first_stage.summary)
     ```

3. **Assess instrument strength.**  
   - Examine the **first-stage F-statistics** reported at the bottom of each first-stage table.  
   - Are your instruments **strong** (F ≥ 10) or **weak** (F < 10)?  
   - Briefly comment on what that implies for your IV estimates.

4. **Compare the 2SLS and OLS results.**  
   - How do the **coefficients** on road variables differ between OLS and IV?  
   - Does the IV estimate suggest that OLS **overstated** or **understated** the causal effect of roads on urban population growth?  
   - What does this tell you about the **direction of endogeneity bias** in the OLS regressions?


In [4]:
# QUESTION 3 — IV (2SLS) Regressions (Table 4: Columns 1, 2, and 6)

# (1) Baseline IV: Δln(CC Pop) ~ all_road_2010_rays  (IV = road_1962_rays)
# your code here

# (2) Baseline + controls + ΔPrefPop endogenous
controls = [
    "larea_cc90",
    "larea_pf05",
    "province_capitalPlus",
    "lcensuspop1982_pref",
    "frac_highedu1982_pref",
    "share_emp_man1982_pref",
]
# your code here

# (6) Radial + Ring Roads together
# your code here

# ---- Display Results ----
# your code here



## Answer for 3.3

### 3) Instrument strength
your answer here

### 4) OLS vs. IV — coefficient comparison and bias direction
your answer here

**Interpretation:**  
your answer here


# Question 4 — Standard Error Sensitivity Analysis

In applied econometrics, the choice of **standard error calculation** can significantly affect statistical inference. Different assumptions about the error structure lead to different standard error estimates, which in turn affect p-values and conclusions about statistical significance.

In this question, you will re-estimate **specification (6)** from Question 3 (the radial+ring IV regression) using **three different standard error assumptions**:

1. **Classical standard errors** — Assume errors are independent and identically distributed (IID)  
2. **HC1 robust standard errors** — Allow for heteroskedasticity but assume independence  
3. **Clustered standard errors** — Allow for correlation within provinces (as used in main analysis)

Focus your analysis on the coefficient for **`all_road_2010_rays`** only.

---

## 4.1 Tasks

1. **Re-estimate specification (6)** using each of the three standard error types above
2. **Report the three standard errors** for `all_road_2010_rays` and note the significance level under each assumption
3. **Identify which standard error assumption is most conservative** (provides largest standard errors)
4. **Provide conclusions** based on this analysis about the appropriateness of different standard error assumptions

In [5]:
# QUESTION 4 — Standard Error Sensitivity Analysis
# Focus on specification (6) and all_road_2010_rays variable only

# Set up the IV regression (specification 6: radial+ring from Question 3)
controls = [
    "larea_cc90",
    "larea_pf05", 
    "province_capitalPlus",
    "lcensuspop1982_pref",
    "frac_highedu1982_pref",
    "share_emp_man1982_pref",
]

# 1) Classical standard errors (IID assumption)
# your code here

# 2) HC1 robust standard errors (heteroskedasticity-robust)
# your code here

# 3) Clustered standard errors (by province)
# your code here

# Extract and print results
# your code here

## Answer to 4.1 Task 4

Your answer here